# MPCC FPGA HIL server

MATLAB sends 14 little-endian `single` values to port 5010. The FPGA HLS IP returns one `single` duty prediction `D` on port 5011.

In [1]:
import sys
from pathlib import Path


search_roots = [
    Path.cwd(),
    Path.cwd() / "PS_notebook",
    Path("/home/xilinx/jupyter_notebooks/PS_notebook"),
]
NOTEBOOK_ROOT = next(
    (root for root in search_roots if (root / "libs").is_dir()),
    None,
)
if NOTEBOOK_ROOT is None:
    raise FileNotFoundError("找不到包含 libs/ 的 PS_notebook 目录")
sys.path.insert(0, str(NOTEBOOK_ROOT))

from libs.mpcc_overlay import (
    INPUT_NAMES,
    MPCCOverlayPredictor,
    load_mpcc_overlay,
)

BIT_PATH = NOTEBOOK_ROOT / "hardware/mpcc_hil.bit"
HWH_PATH = NOTEBOOK_ROOT / "hardware/mpcc_hil.hwh"
overlay, mpcc_ip = load_mpcc_overlay(BIT_PATH, HWH_PATH)
accelerator = MPCCOverlayPredictor(mpcc_ip)

print("BIT:", BIT_PATH)
print("HWH:", HWH_PATH)
print("IP inputs:", INPUT_NAMES)


BIT: /home/xilinx/jupyter_notebooks/hardware/mpcc_hil.bit
HWH: /home/xilinx/jupyter_notebooks/hardware/mpcc_hil.hwh
IP inputs: ('i_L', 'i_ref', 'V_in', 'Ts', 'L_in', 'V_o', 'theta_pll', 'A3', 'A5', 'A7', 'phi3', 'phi5', 'phi7', 'use_harmonic')


In [2]:
from libs.tcp_cosim_utils import initialize_server


def process_hil_frame(frame):
    return accelerator.predict(frame)


hil_server = initialize_server(
    process_function=process_hil_frame,
    namespace=globals(),
    namespace_key="hil_server",
    input_port=5010,
    output_port=5011,
    data_type="single",
    batch_size=14,
    output_batch_size=1,
    processing_mode="frame",
)


Output channel 'result' listening on port 5011
Input server listening on port 5010
Co-simulation server configured
Input: port=5010, type=single, batch=14, bytes=56
Output: port=5011, type=single, batch=1, mode=frame


In [3]:
print("Predictions:", accelerator.call_count)
print(f"Average FPGA transaction: {accelerator.average_hardware_us:.2f} us")


Predictions: 0
Average FPGA transaction: 0.00 us


Simulink Send connected from ('134.226.169.95', 40928)
Simulink Send connected from ('134.226.169.95', 36112)
Simulink Receive [result] connected from ('134.226.169.95', 39918)
Simulink Send connected from ('134.226.169.95', 54040)
Simulink Receive [result] connected from ('134.226.169.95', 55424)
Simulink Send connected from ('134.226.169.95', 56714)
Simulink Receive [result] connected from ('134.226.169.95', 40288)
Simulink Send connected from ('134.226.169.95', 50190)
Simulink Send connected from ('134.226.169.95', 60474)
Simulink Receive [result] connected from ('134.226.169.95', 44130)
Simulink Send connected from ('134.226.169.95', 41612)
Simulink Receive [result] connected from ('134.226.169.95', 40084)
Simulink Send connected from ('134.226.169.95', 43770)
Simulink Receive [result] connected from ('134.226.169.95', 59322)
Simulink Send connected from ('134.226.169.95', 51178)
Simulink Receive [result] connected from ('134.226.169.95', 55322)
Simulink Send connected from ('134.2